# 06. Pruning, quantizing, both, and the final curves

Pruning without retraining (CORP, GRAIL, VBP), pruned models taken to low bits, our per-block allocation with its three curves, and the checks around them. The last sections report some extra metrics we took into consideration besides size and accuracy: MACs, BOPs, latency, memory and time to compress.


With `RERUN = False` everything is read from `outputs/`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import glob
import os
import subprocess
import sys

REPO = Path.cwd()
THIRD = Path(os.environ.get("THIRD_PARTY", REPO.parent / "third_party"))   # scripts/setup_third_party.sh
PY = sys.executable                                     # this environment runs our scripts
os.environ.setdefault("PY", PY)
os.environ.setdefault("ADALOG_PY", "python")            # the published methods' environment, see the README

def sh(cmd):
    """Run a command from the repository root, show its output as it comes, and stop if it fails."""
    with subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"exit code {p.returncode}: {cmd}")

def latest(pattern):
    """The newest file under ../third_party that matches `pattern`."""
    found = sorted(glob.glob(str(THIRD / pattern)), key=os.path.getmtime)
    if not found:
        raise FileNotFoundError(pattern)
    return found[-1]

def baselines():
    """The dense baselines of notebook 02, as {model: {"size_mb": ..., "accuracy": ...}}."""
    b = json.loads((Path("outputs") / "baselines_summary.json").read_text())
    return {k: {"size_mb": b["size_mb"][k], "accuracy": a} for k, a in b["accuracy"].items()}

def checkpoint_of_run(name):
    """The checkpoint a FIMA-Q or APHQ-ViT run saved after block reconstruction, read from its log."""
    folder = THIRD / ("APHQ-ViT-protocol" if name.startswith("APHQ") else "FIMA-Q-protocol")
    log = (folder / f"run_{name.replace(' ', '_')}.log").read_text(errors="replace")
    saved = [l.split("Saving checkpoint to ")[1].strip() for l in log.splitlines() if "Saving checkpoint to" in l and "optimsize" in l]
    return str(folder / saved[-1])

RERUN = False          # True calls the scripts; each FIMA-Q search takes 40 min to 2 h
OUT = Path("outputs")
SETS = ["ImageNet-1k val", "ImageNet-V2", "ImageNet-R"]
load = lambda f: json.loads((OUT / f).read_text())
top1 = lambda e, s="ImageNet-1k val": e["accuracy"][s]["top1"]
def row(name, e, ref=None):
    accs = " / ".join(f"{top1(e, s):6.2f}" for s in SETS)
    d = f"   vs ref {top1(e) - top1(ref):+6.2f}" if ref else ""
    size = f"{e['size_mb']:7.2f} MB" if e.get("size_mb") else "      -   "   # points scored in memory have no file
    print(f"{name:44s} {size}  {accs}{d}")
base = baselines()
pub = load("published_summary.json")
ti_int8, s_int8 = base["DeiT-Ti INT8"], base["DeiT-S INT8"]
def limits(ax):
    ax.axhline(top1(ti_int8), ls=":", c="grey", lw=1.2, label="DeiT-Ti INT8 (limit)")

## Pruning alone

63 models in FP16 and FP32 (`scripts/prune/prune_points.py`, `deitc/pruning.py`; VBP and GRAIL run the authors' code from `third_party/`). GRAIL runs with the 128 images of its paper and with 4 000. DeiT-S INT8 is drawn for scale: no pruned model in FP16 beats an unpruned one of the same size or smaller.

In [ ]:
pr = load("pruned_summary.json")   # the pruned models are rebuilt in notebook 04
fig, ax = plt.subplots(figsize=(7, 4.5)); limits(ax)
groups = {}
for k, v in pr.items():
    if v.get("size_mb") and "fp16" in str(v.get("file", "fp16")).lower():
        g = k.rsplit("_s", 1)[0] + ("_n4000" if "n4000" in k else "")
        groups.setdefault(g, []).append((v["size_mb"], top1(v)))
for g, pts in sorted(groups.items()):
    x, y = zip(*sorted(pts)); ax.plot(x, y, "o:", ms=3, alpha=0.6, label=g)
ax.plot(s_int8["size_mb"], top1(s_int8), "k*", ms=10, label="DeiT-S INT8")
ax.set(xlabel="file size (MB)", ylabel="top-1 on val (%)", title="Pruning without retraining")
ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.3); plt.show()

## Pruned models at low bits

Round-to-nearest at W3 and W4 on twelve pruned models, and FIMA-Q weights-only on seven (`scripts/prune/combine_points.py`, `methods/fimaq/run.sh --pruned-weights`).

In [ ]:
if RERUN:
    for base in [f"{m}_s{s}" for m in ("corp-both", "corp-mlp") for s in (10, 20, 30, 40)] + [f"grail-wanda_s{s}_n4000" for s in (10, 20, 30, 40)]:
        sh(f'"{PY}" scripts/prune/combine_points.py rtn {base} --bits 3 4')
    for base, b in [("corp-both_s10", 3), ("corp-both_s20", 3), ("corp-both_s30", 3), ("corp-mlp_s20", 3),
                    ("grail-wanda_s20_n4000", 3), ("corp-both_s30", 4), ("corp-both_s40", 4)]:
        name, out = f"{base}+fimaq_w{b}", f"outputs/artifacts/{base}_fimaq_w{b}.safetensors"
        sh(f'SCORE_OUT=combined_check.json bash methods/fimaq/run.sh {b}bit/w{b}a32.py "{name}" '
           f'--pruned-weights "{REPO}/outputs/artifacts/{base}_fp32.safetensors"')
        sh(f'"{PY}" scripts/quantize/pack_published.py --method fimaq --bits {b} --ckpt "{checkpoint_of_run(name)}" --out {out}')
        r = load("combined_check.json")[f"{name}, in the method's forward"]
        sh(f'"{PY}" scripts/prune/combine_points.py file {base} fimaq {b} {out} --seconds {r["calibration_s"] + r["block_reconstruction_s"]}')
for k, v in sorted(load("combined_summary.json").items(), key=lambda kv: kv[1].get("size_mb") or 0):
    row(k, v)

## Our allocation: bits only, pruning only, and both

`scripts/allocate/allocate.py` takes each layer's bits and each block's pruning from the KL on calibration images in `sensitivity_summary.json`, under a target size, and every plan is built and scored as a whole model: FIMA-Q with per-layer bits (q), CORP joint in FP16 (p), and CORP then FIMA-Q (both). Uniform FIMA-Q and DeiT-Ti INT8 are drawn for reference.

In [ ]:
if RERUN:
    allocate = f'"{PY}" scripts/allocate/allocate.py'
    plans = [json.loads(f.read_text()) for f in sorted((OUT / "allocations").glob("*.json")) if not f.name.endswith(".bits.json")]
    for p in plans:   # the ablation's plan was written by hand and is kept as it is
        if p["curve"] in ("q", "p", "both"):
            sh(f'{allocate} plan {p["name"]} --curve {p["curve"]} --target-mb {p["target_mb"]} --signal {p["signal"]}')
    sh(f'{allocate} build ' + " ".join(p["name"] for p in plans if p["curve"] != "q"))   # pruning-only models are scored here
    for p in plans:
        if p["curve"] == "q":
            sh(f'{allocate} quantize {p["name"]}')
        elif p["curve"] in ("both", "ablation"):
            sh(f'{allocate} quantize {p["name"]} --base {p["name"]}')
al = load("allocated_summary.json")
fig, ax = plt.subplots(figsize=(7, 4.5)); limits(ax)
for curve, c in (("q", "#55A868"), ("both", "#DD8452")):
    pts = sorted((v["size_mb"], top1(v)) for v in al.values() if v.get("curve") == curve)
    x, y = zip(*pts); ax.plot(x, y, "o-", c=c, label={"q": "ours, bits only", "both": "ours, pruning and bits"}[curve])
uni = sorted((v["size_mb"], top1(v)) for k, v in pub.items() if k.startswith("FIMA-Q W") and "A" not in k[7:] and v.get("size_mb"))
x, y = zip(*uni); ax.plot(x, y, "s:", c="#55A868", alpha=0.5, mfc="none", label="FIMA-Q uniform")
ab = al.get("abl-b0attn-s30")
if ab: ax.plot(ab["size_mb"], top1(ab), "x", c="black", label="block 0 attention unpruned")
ax.set(xlim=(4.5, 15), xlabel="file size (MB)", ylabel="top-1 on val (%)", title="Allocated curves")
ax.legend(fontsize=7); ax.grid(alpha=0.3); plt.show()
for k, v in sorted(al.items(), key=lambda kv: (kv[1].get("curve"), kv[1]["size_mb"])):
    row(f"{v.get('curve'):9s} {k}", v)

## Paired comparisons behind the claims

Each claim compares two models on the same images. The paired difference and its 95 % interval are given on val, V2 and R, and a claim needs the interval to exclude zero on val with the same sign on the other two sets.

In [ ]:
from deitc.evaluate import paired_diff_ci
Z = {}
for f in ["allocated", "baselines", "published", "a8", "combined", "pruned", "w8a8"]:
    z = np.load(OUT / f"{f}_per_image.npz")
    Z.update({k: z[k] for k in z.files})
PAIRS = [("both6.7", "q6.7"), ("both7.4", "q7.4"), ("both7.7", "q7.7"), ("both8.1", "q8.1"), ("both10", "q10"),
         ("q6.7", "DeiT-Ti INT8"), ("FIMA-Q W2", "DeiT-Ti INT8"), ("both6", "DeiT-Ti INT8"), ("both5", "DeiT-Ti INT4"), ("FIMA-Q W2", "both6"),
         ("q14", "DeiT-S FP32"), ("q10", "DeiT-S FP32"), ("q14", "FIMA-Q W4"), ("q10", "FIMA-Q W3"), ("q8.1", "FIMA-Q W3"),
         ("abl-b0attn-s30", "both6.7"), ("abl-b0attn-s30", "q6.7"),
         ("q7.4 A8", "q7.4"), ("q6.7 A8", "q6.7"), ("both6.7 A8", "both6.7"), ("q14 A8", "q14"), ("FIMA-Q W3 A8 sim", "FIMA-Q W3"),
         ("p40", "DeiT-S INT8"), ("p40 W8A8", "DeiT-S W8A8 torchao"), ("corp-both_s10 W8A8", "DeiT-S W8A8 torchao"), ("p40 W8A8", "p40")]
print(f"{'':38s} " + " ".join(f"{s:>18s}" for s in SETS))
for a, b in PAIRS:
    cells = []
    for s in SETS:
        if f"{s}|{a}" in Z and f"{s}|{b}" in Z:
            d, (lo, hi) = paired_diff_ci(Z[f"{s}|{a}"].astype(float), Z[f"{s}|{b}"].astype(float))
            cells.append(f"{d:+6.2f} [{lo:+.2f},{hi:+.2f}]")
        else:
            cells.append(f"{'missing':>18s}")
    print(f"{a + ' vs ' + b:38s} " + " ".join(cells))

## Compression order

Quantizing first and pruning second, against the usual order at the same sizes (`scripts/prune/quantize_then_prune.py`). Only the usual order gets a fresh FIMA-Q search on the pruned weights, so a loss for this order is weak evidence.

In [ ]:
if RERUN:
    sh(f'"{PY}" scripts/prune/quantize_then_prune.py 20 30')
comb = load("combined_summary.json")
for k, v in load("order_summary.json").items():
    pct = k.split()[-2]
    usual = comb.get(f"corp-both_s{pct}+fimaq_w3")
    row(k, v, usual)

## Weights and activations

FIMA-Q as published on pruned models (`combined_wa.json`), 8-bit activations on our models and on FIMA-Q's files (`scripts/quantize/a8_score.py`), and the pruned models stored as torchao W8A8 (`scripts/quantize/w8a8_torchao.py --pruned`).

In [ ]:
if RERUN:
    for k in load("combined_wa.json"):   # FIMA-Q as published on CORP joint models
        base, wa = k.split("+fimaq_")
        sh(f'SCORE_OUT=combined_wa.json bash methods/fimaq/run.sh {wa[1]}bit/best.py "{k}" --pruned-weights "{REPO}/outputs/artifacts/{base}_fp32.safetensors"')
    pruned = ["p40", "p37.5", "p35", "p32.5", "p30", "corp-both_s10", "corp-both_s20", "corp-both_s30"]
    sh(f'"{PY}" scripts/quantize/w8a8_torchao.py --pruned {" ".join(pruned)} --pack-only')
    pairs = [f'"FIMA-Q W{b} A8 sim"=outputs/artifacts/fimaq_w{b}.safetensors' for b in (3, 4)]
    pairs += [f'"{n} A8"=outputs/artifacts/{n}_fimaq.safetensors' for n in ("q7.4", "q8.1", "q6.7", "q7.7", "q10", "q14", "both6.7", "both7.4", "both8.1", "both10")]
    pairs += [f'"{n} W8A8"=outputs/artifacts/{n}_w8a8_torchao.safetensors' for n in pruned]
    sh(f'"{PY}" scripts/quantize/a8_score.py ' + " ".join(pairs))
print("FIMA-Q as published on pruned models")
for k, v in load("combined_wa.json").items():
    row(k, v, pub["FIMA-Q " + k.rsplit("_", 1)[1].upper()])
print("\n8-bit activations, against the same weights with float activations")
a8 = load("a8_summary.json")
for k, v in a8.items():
    src = al.get(k.split()[0]) or pub.get(k.replace(" A8 sim", ""))
    row(k, v, src)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5)); limits(ax)
wa = [(v["size_mb"], top1(v)) for k, v in pub.items() if v.get("run") == "as published" and v.get("size_mb")]
ax.plot(*zip(*wa), "s", mfc="none", c="#4C72B0", label="published, as published (W=A)")
ax.plot(*zip(*[(v["size_mb"], top1(v)) for v in a8.values()]), "o", c="#DD8452", alpha=0.7, label="8-bit activations")
ax.set(xlabel="file size (MB)", ylabel="top-1 on val (%)", title="Weights and activations quantized")
ax.legend(fontsize=7); ax.grid(alpha=0.3); plt.show()

## Costs besides size and accuracy

MACs and BOPs for every point (`scripts/measure/measure_costs.py`; BOPs count an activation left at full precision as 32 bits), latency and peak memory on the RTX 4070 (`scripts/measure/measure_latency.py`; the low-bit kernels available were not built for this architecture, so those timings say little about low-bit hardware), and time to compress.

In [ ]:
if RERUN:
    sh(f'"{PY}" scripts/measure/measure_costs.py')
    sh(f'"{PY}" scripts/measure/measure_latency.py')            # run with nothing else on the GPU
    sh(f'"{PY}" scripts/measure/measure_latency.py --pruned')
costs = load("costs_summary.json")
print(costs["convention"])
for k, v in costs["points"].items():
    print(f"{k:44s} " + "  ".join(f"{f}={x:.4g}" if isinstance(x, float) else f"{f}={x}" for f, x in v.items() if f not in ("model",)))

In [ ]:
for f in ["latency_summary.json", "latency_pruned_summary.json"]:
    print("==", f)
    for k, v in load(f)["results"].items():
        if not isinstance(v, dict):   # the check that W8A8 computes with integer kernels
            print(f"{k:52s} {', '.join(v)}")
            continue
        print(f"{k:52s} {v['median_ms']:8.2f} ms  {v['images_per_s']:8.0f} img/s  " + "  ".join(f"{a}={b:.0f}" for a, b in v.items() if "mib" in a.lower()))

In [ ]:
print("Time to compress (seconds)")
for f in ["published_summary.json", "pruned_summary.json", "allocated_summary.json"]:
    for k, v in load(f).items():
        t = {a: round(b) for a, b in v.items() if isinstance(b, (int, float)) and (a.endswith("_s") or "seconds" in a)}
        if t: print(f"{k:44s} {t}")

## Under shift

Accuracy on V2 and R against val for every weights-only model. Points under the line lose more under shift than their val accuracy suggests.

In [ ]:
allpts = {**{k: v for k, v in pub.items() if v.get("size_mb")}, **al, **load("combined_summary.json")}
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, s in zip(axs, SETS[1:]):
    x = [top1(v) for v in allpts.values()]; y = [top1(v, s) for v in allpts.values()]
    ax.scatter(x, y, s=10, alpha=0.6)
    a, b = np.polyfit(x, y, 1); xs = np.linspace(min(x), max(x), 2); ax.plot(xs, a * xs + b, "k:", lw=1)
    ax.set(xlabel="top-1 on val (%)", ylabel=f"top-1 on {s} (%)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()